In [1]:
# ===============================================
#              RAG com Qdrant (substitui FAISS)
# ===============================================

# 1. Instalações necessárias (rode uma vez no terminal ou descomente)
# !pip install qdrant-client sentence-transformers gpt4all pymupdf numpy

import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
import uuid
from gpt4all import GPT4All

c:\Users\Yuan\miniconda3\envs\wh\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'qdrant_client'

In [ ]:
# Evita erro de distributed em alguns ambientes
import torch
import torch.distributed as dist
if not hasattr(dist, 'is_initialized'):
    dist.is_initialized = lambda: False

# ===============================================
# Funções básicas (igual ao seu original)
# ===============================================

def extrair_texto_do_pdf(caminho_pdf):
    doc = fitz.open(caminho_pdf)
    texto_completo = ""
    for pagina in doc:
        texto_completo += pagina.get_text()
    doc.close()
    return texto_completo

def dividir_em_chunks(texto, tamanho_chunk=500, sobreposicao=50):
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fim = inicio + tamanho_chunk
        chunks.append(texto[inicio:fim])
        inicio += tamanho_chunk - sobreposicao
    return chunks

def criar_embeddings(chunks, nome_modelo='./bge-small-en-v1.5'):
    model = SentenceTransformer(nome_modelo)
    print(f"Criando embeddings para {len(chunks)} chunks...")
    embeddings = model.encode(chunks, show_progress_bar=True)
    print("Embeddings criados!")
    return embeddings

# ===============================================
# Parte que muda: Indexação com Qdrant
# ===============================================

# Carrega os arquivos já gerados (ou gere novamente se quiser)
embeddings = np.load("embeddings.npy")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"Carregados {len(chunks)} chunks e {embeddings.shape[0]} vetores de dimensão {embeddings.shape[1]}")

# Conecta no Qdrant (em memória para teste rápido)
client = QdrantClient(location=":memory:")  
# Para persistir em disco (recomendado):
# client = QdrantClient(path="./qdrant_db_meu_rag")

collection_name = "rag_beesdata_delta_share"

# Cria/recria a coleção
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=embeddings.shape[1],     # 384 para bge-small-en-v1.5
        distance=Distance.COSINE      # melhor para embeddings de texto
    )
)

# Prepara e insere os pontos
points = []
for vec, texto in zip(embeddings, chunks):
    point = PointStruct(
        id=str(uuid.uuid4()),
        vector=vec.tolist(),
        payload={"conteudo": texto}   # nomeei de "conteudo" para ficar claro
    )
    points.append(point)

client.upsert(collection_name=collection_name, points=points)
print(f"↑ {len(points)} vetores inseridos no Qdrant com sucesso.")

# ===============================================
# Busca semântica (substitui a do FAISS)
# ===============================================

model = SentenceTransformer('./bge-small-en-v1.5')

def buscar_contexto(pergunta, k=4):
    emb_pergunta = model.encode([pergunta])[0].tolist()
    
    search_result = client.search(
        collection_name=collection_name,
        query_vector=emb_pergunta,
        limit=k,
        with_payload=True
    )
    
    resultados = [hit.payload["conteudo"] for hit in search_result]
    scores = [hit.score for hit in search_result]
    
    print("Scores das buscas:", [f"{s:.4f}" for s in scores])
    return resultados

# ===============================================
# Monta prompt e responde com LLM local
# ===============================================

def montar_prompt(pergunta, contextos):
    contexto_texto = "\n\n".join(contextos)
    return f"""
<|system|>
Responda apenas com base no contexto abaixo.
Se a resposta não estiver no contexto, diga que não encontrou.
Responda em português.
</s>

<|user|>
CONTEXTO:
{contexto_texto}

PERGUNTA:
{pergunta}
</s>

<|assistant|>
"""

# Carrega o LLM local (ajuste o caminho se necessário)
llm = GPT4All(
    r"./tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    device="gpu"   # ou "cpu" se não tiver GPU compatível
)

def responder_rag(pergunta):
    contextos = buscar_contexto(pergunta, k=4)
    prompt = montar_prompt(pergunta, contextos)
    
    with llm.chat_session():
        resposta = llm.generate(
            prompt,
            max_tokens=800,
            temp=0.3
        )
    return resposta

# ===============================================
# Teste rápido
# ===============================================

if __name__ == "__main__":
    pergunta = "What is the Data Ops L2 process?"
    print("\nPergunta:", pergunta)
    print("\nResposta gerada:\n")
    print(responder_rag(pergunta))